In [5]:
import pandas as pd
import yaml
from sqlalchemy import create_engine

In [6]:
with open('config.yml', 'r') as f:
    config = yaml.safe_load(f)

config_etl = config['ETL_PRO']

url = (
    f"{config_etl['drivername']}://"
    f"{config_etl['user']}:{config_etl['password']}@"
    f"{config_etl['host']}:{config_etl['port']}/"
    f"{config_etl['dbname']}"
)

etl_conn = create_engine(url)

In [ ]:
hecho_servicios = pd.read_sql_table('hecho_servicios', etl_conn)
hecho_novedades = pd.read_sql_table('hecho_novedades', etl_conn)

dim_fecha = pd.read_sql_table('dim_fecha', etl_conn)
dim_hora = pd.read_sql_table('dim_hora', etl_conn)
dim_proveedor = pd.read_sql_table('dim_proveedor', etl_conn)
dim_mensajero = pd.read_sql_table('dim_mensajero', etl_conn)
dim_sede = pd.read_sql_table('dim_sede', etl_conn)
dim_novedades = pd.read_sql_table('dim_novedades', etl_conn)

## Pregunta 1

In [9]:
consulta = """
SELECT
    df.mes,
    COUNT(*) AS total_servicios
FROM hecho_servicios hs
JOIN dim_fecha df
    ON hs.key_dim_fecha_solicitud = df.key_fecha
GROUP BY df.mes
ORDER BY total_servicios DESC;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,mes,total_servicios
0,5,4725
1,7,4550
2,4,4480
3,8,4304
4,6,4184
5,3,3337
6,2,2479
7,1,296
8,12,25
9,9,21


## Pregunta 2

In [10]:
consulta = """
SELECT
    df.dia_semana,
    COUNT(*) AS total_servicios
FROM hecho_servicios hs
JOIN dim_fecha df
    ON hs.key_dim_fecha_solicitud = df.key_fecha
GROUP BY df.dia_semana
ORDER BY total_servicios DESC;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,dia_semana,total_servicios
0,2,5398
1,5,5281
2,4,5161
3,3,4961
4,1,4308
5,6,2481
6,7,840


## Pregunta 3

In [11]:
consulta = """
SELECT
    dh.hora,
    COUNT(*) AS total_servicios
FROM hecho_servicios hs
JOIN dim_hora dh
    ON hs.key_dim_hora_solicitud = dh.key_hora
GROUP BY dh.hora
ORDER BY total_servicios DESC;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,hora,total_servicios
0,9,3391
1,11,3295
2,10,2932
3,15,2658
4,8,2636
5,14,2573
6,16,2124
7,12,1721
8,13,1465
9,17,1357


## Pregunta 4

In [12]:
consulta = """
SELECT
    dp.id_proveedor,
    df.mes,
    COUNT(*) AS total_servicios
FROM hecho_servicios hs
JOIN dim_proveedor dp
    ON hs.key_dim_proveedor = dp.key_proveedor
JOIN dim_fecha df
    ON hs.key_dim_fecha_solicitud = df.key_fecha
GROUP BY
    dp.id_proveedor,
    df.mes
ORDER BY
    dp.id_proveedor,
    df.mes;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,id_proveedor,mes,total_servicios
0,2,1,1
1,2,2,16
2,2,3,5
3,2,4,2
4,2,5,2
...,...,...,...
102,25,5,155
103,25,6,155
104,25,7,178
105,25,8,175


## Pregunta 5

In [13]:
consulta = """
SELECT
    dm.id_operaciones,
    COUNT(*) AS servicios_realizados
FROM hecho_servicios hs
JOIN dim_mensajero dm
    ON hs.key_mensajero_1 = dm.key_mensajero
GROUP BY dm.id_operaciones
ORDER BY servicios_realizados DESC;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,id_operaciones,servicios_realizados
0,30,2439
1,29,1553
2,15,1514
3,25,1456
4,31,1352
5,16,1333
6,41,1329
7,42,1254
8,22,1252
9,28,1228


## Pregunta 6

In [14]:
consulta = """
SELECT
    dp.id_proveedor,
    ds.sede_id,
    COUNT(*) AS total_servicios
FROM hecho_servicios hs
JOIN dim_proveedor dp
    ON hs.key_dim_proveedor = dp.key_proveedor
JOIN dim_sede ds
    ON hs.key_sede = ds.key_sede
GROUP BY
    dp.id_proveedor,
    ds.sede_id
ORDER BY
    dp.id_proveedor,
    total_servicios DESC;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,id_proveedor,sede_id,total_servicios
0,11,15,974
1,11,17,962
2,11,16,855
3,11,26,331
4,11,22,208
5,11,23,124
6,11,34,34
7,11,25,33
8,11,35,33
9,11,33,14


## Pregunta 7

In [15]:
consulta = """
SELECT
AVG(
(fecha_finalizado_completo + hora_finalizado_completo)
-
(fecha_solicitud + hora_solicitud)
) AS tiempo_promedio
FROM trans_servicio;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,tiempo_promedio
0,0 days 12:38:31.770393


## Pregunta 8

In [16]:
consulta = """
SELECT

AVG(
(fecha_mensajero_asignado + hora_mensajero_asignado)
-
(fecha_iniciado + hora_iniciado)
) AS iniciado_asignado,

AVG(
(fecha_recogido_mensajero + hora_recogido_mensajero)
-
(fecha_mensajero_asignado + hora_mensajero_asignado)
) AS asignado_recogido,

AVG(
(fecha_entregado + hora_entregado)
-
(fecha_recogido_mensajero + hora_recogido_mensajero)
) AS recogido_entregado,

AVG(
(fecha_finalizado_completo + hora_finalizado_completo)
-
(fecha_entregado + hora_entregado)
) AS entregado_cerrado

FROM trans_servicio;
"""
resultado = pd.read_sql(consulta, etl_conn)
resultado

,iniciado_asignado,asignado_recogido,recogido_entregado,entregado_cerrado
0,0 days 02:29:03.098227,0 days 01:47:52.625639,0 days 01:47:55.811086,0 days 04:54:33.785686


## Pregunta 9

In [17]:
consulta = """
SELECT
    dn.key_novedad,
    COUNT(*) AS total_novedades
FROM hecho_novedades hn
JOIN dim_novedades dn
    ON hn.key_novedad = dn.key_novedad
GROUP BY dn.key_novedad
ORDER BY total_novedades DESC;
"""
resultado = pd.read_sql(consulta, etl_conn)
resultado

,key_novedad,total_novedades
0,26,1
1,47,1
2,46,1
3,15,1
4,14,1
5,20,1
6,17,1
7,16,1
